In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti
from myPythonTools.TopoAna import rdremap as RM
from myPythonTools.TopoAna import rdridgelist as RL
from myPythonTools.TopoAna import cube_panels as CP


import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)
importlib.reload(RM)
importlib.reload(RL)
importlib.reload(CP)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
def ridge_scale_from_fields(fields, sigmas_km, dx_km=3.336, gamma=0.75):
    """
    fields[k] : 2-D array [y, x], already blurred to width sigmas_km[k] (km).
    Returns, per pixel, at the width where the ridge response is strongest:
      sig_best : that width (km)
      R1_best  : fair ridge strength, about 0.35 x ridge height (m)
      th_cross : cross-ridge direction, degrees from +x toward +y, in [0, 180)
    """
    ridge_crit = 0.95 
    shape    = fields[0].shape
    R_best   = np.full(shape, -np.inf)
    R1_best  = np.full(shape, np.nan)
    sig_best = np.full(shape, np.nan)
    th_cross = np.full(shape, np.nan)
    for F, s_km in zip(fields, sigmas_km):
        s = s_km / dx_km                                   # blur width in cells
        Fy, Fx = np.gradient(F)                            # slopes (axis 0 = y, axis 1 = x)
        Lyy = np.gradient(Fy, axis=0)                      # curvatures
        Lxx = np.gradient(Fx, axis=1)
        Lxy = np.gradient(Fx, axis=0)
        half_tr = 0.5 * (Lxx + Lyy)
        rad     = np.sqrt(0.25 * (Lxx - Lyy)**2 + Lxy**2)
        lam_q   = half_tr - rad                            # curvature across the crest
        lam_p   = half_tr + rad                            # curvature along the crest
        ridge_like = (lam_q < 0) & (np.abs(lam_p) < ridge_crit * np.abs(lam_q))
        R  = np.where(ridge_like, (s**2)**gamma * (-lam_q), -np.inf)   # chooses the width
        R1 = (s**2) * (-lam_q)                                          # compares strengths
        better   = R > R_best
        R_best   = np.where(better, R, R_best)
        R1_best  = np.where(better, R1, R1_best)
        sig_best = np.where(better, s_km, sig_best)
        th_p     = 0.5 * np.degrees(np.arctan2(2 * Lxy, Lxx - Lyy))
        th_cross = np.where(better, (th_p + 90.) % 180., th_cross)
    return sig_best, R1_best, th_cross

In [ ]:
from scipy.ndimage import gaussian_filter

snapfile = "/glade/work/juliob/Topo/NCARTopoJTB-fire/cases/ne30pg3_Sco100_Rw050_Wm1_smooth_dev_snaps_v206_e04/output/topo_smooth_snapshots_gmted2010_modis_bedmachine_nc3000_Co060.nc"
topofile = "/glade/work/juliob/Topo/NCARTopoJTB-fire/cases/ne30pg3_Sco100_Rw050_Wm1_smooth_dev_snaps_v206_e04/output/topo_smooth_gmted2010_modis_bedmachine_nc3000_Co060.nc"

# --- files ---------------------------------------------------------------

nc=3000

topo = xr.open_dataset(topofile)

snap = xr.open_dataset(snapfile)
sig  = snap.sigma_km.values
#S    = snap.terr_snap.values.reshape(len(sig), 6, nc, nc)
S    = snap.dev_snap.values.reshape(len(sig), 6, nc, nc)
sm   = np.reshape(topo['terr_sm'].values,  (6, nc, nc))
dev  = np.reshape(topo['terr_dev'].values, (6, nc, nc))
print("cycles:", snap.cycle.values, "  sigma (km):", np.round(sig, 1), "  sigma_final:", snap.sigma_final_km)

tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )

# --- sanity: blurrier snapshots move away from raw and toward terr_sm -----
#for k in range(len(sig)):
#    print(f"{sig[k]:6.1f} km   rms(snap-raw) {np.std(S[k]-(sm+dev)):7.1f} m   rms(snap-terr_sm) {np.std(S[k]-sm):7.1f} m")


# --- sanity: blurrier snapshots move away from dev and shrink toward zero ---
print(f"unblurred dev: rms {np.std(dev):7.1f} m")
for k in range(len(sig)):
    print(f"{sig[k]:6.1f} km   rms(snap-dev) {np.std(S[k]-dev):7.1f} m   rms(snap) {np.std(D[k]):7.1f} m")



"""
# --- region: Tennessee on face 4 (same as before) -------------------------
p = 3
m = CP.in_box(tlons[p], tlats[p], [268., 285.], [30., 41.])
yy, xx = np.nonzero(m)
y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1
"""
# --- region: SW US on face 4 (same as before) -------------------------
p = 3
m = CP.in_box(tlons[p], tlats[p], [268.-30, 285.-30], [30., 41.])
yy, xx = np.nonzero(m)
y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1

# (a) blurred fields from the smoother snapshots
#fields_snap = [S[k, p, y0:y1, x0:x1] - sm[p, y0:y1, x0:x1] for k in range(len(sig))]
fields_snap = [S[k, p, y0:y1, x0:x1] for k in range(len(sig))]

# (b) the same widths with gaussian_filter on dev (crop with a margin, filter, then trim)
pad = int(4 * sig.max() / 3.336) + 1
Y0, Y1, X0, X1 = max(y0-pad, 0), min(y1+pad, nc), max(x0-pad, 0), min(x1+pad, nc)
D = dev[p, Y0:Y1, X0:X1]
fields_gauss = [gaussian_filter(D, s/3.336)[y0-Y0:y1-Y0, x0-X0:x1-X0] for s in sig]

sA, rA, tA = ridge_scale_from_fields(fields_snap,  sig, gamma=1.)
sB, rB, tB = ridge_scale_from_fields(fields_gauss, sig, gamma=1.)

# --- side by side, clear ridges only (R1 ~ 0.35 x ridge height) ------------
fig, ax = plt.subplots(1, 3, figsize=(27, 8))
#for a, s_, r_, title in [(ax[0], sA, rA, 'smoother snapshots - terr_sm'),
for a, s_, r_, title in [(ax[0], sA, rA, 'smoother snapshots (dev)'),
                         (ax[1], sB, rB, 'gaussian_filter(dev)')]:
    im = a.imshow(np.where(r_ > 50., s_, np.nan), origin='lower', cmap='viridis',
                  vmin=sig.min(), vmax=sig.max(), interpolation='nearest')
    a.set_title(title); plt.colorbar(im, ax=a, shrink=0.6, label='best sigma (km)')
both = (rA > 50.) & (rB > 50.)
ax[2].hist2d(sB[both], sA[both], bins=[np.append(sig, sig[-1]*1.5)]*2, cmap='Greys')
ax[2].set_xlabel('gaussian_filter(dev): best sigma'); ax[2].set_ylabel('snapshots: best sigma')
ax[2].set_title('agreement (diagonal = same scale)')
print("same scale at", np.mean(sA[both] == sB[both]).round(2), "of clear-ridge pixels")